# Clase 4 — Calidad de datos y limpieza

**Laboratorio de Modelo Analítico para la Toma de Decisiones**
ISTEA · 2026-2C · División A

---

Ningún dataset de producción llega limpio. Limpiar no es un trámite previo al
análisis: **es una cadena de decisiones que cambian el resultado**. Cada fila que
borrás es una decisión sobre a quién dejás afuera.

Lo que se evalúa en el TP1 no es que limpies igual que la cátedra. Es que puedas
decir **qué hiciste, con qué criterio y a cuántas filas afectó**. Eso es la bitácora,
y vale el 25 % del trabajo.

| Parte | Qué es | Cuándo |
| --- | --- | --- |
| **A** | Limpieza de ElecCommSphere, paso a paso | Lo hacemos juntos |
| **B** | Las cinco revisiones sobre **tu** dataset | Segunda hora, en equipo |

> **La regla de oro de hoy: primero medir, después tocar.** Si limpiás antes de
> contar, perdés el número de «antes» y la bitácora se vuelve imposible de escribir.

---
# PARTE A · ElecCommSphere

In [ ]:
import pandas as pd
import numpy as np

tx  = pd.read_csv("eleccommsphere_transacciones.csv", parse_dates=["fecha"])
cli = pd.read_csv("eleccommsphere_clientes.csv")

print("Transacciones:", tx.shape)
print("Clientes     :", cli.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'eleccommsphere_transacciones.csv'

## A1 · Medir: el informe de calidad

Antes de tocar nada, una foto del estado del archivo. Esta tabla va **tal cual** en
el TP1: es evidencia, no un paso intermedio.

In [ ]:
def informe_calidad(df):
    return pd.DataFrame({
        "tipo":    df.dtypes.astype(str),
        "nulos":   df.isna().sum(),
        "% nulos": (df.isna().mean() * 100).round(2),
        "únicos":  df.nunique(),
    })

informe_calidad(tx)

,tipo,nulos,% nulos,únicos
id_transaccion,int64,0,0.00,26130
id_cliente,int64,0,0.00,3301
fecha,datetime64[ns],0,0.00,541
canal,object,0,0.00,4
categoria,object,0,0.00,17
monto,float64,659,2.50,25464
unidades,int64,0,0.00,3
descuento_pct,float64,0,0.00,6
metodo_pago,object,397,1.51,4
busquedas_sesion,int64,0,0.00,12


### Cómo se lee

**La columna `% nulos` importa más que `nulos`.** Doscientos nulos en una tabla de
mil filas es un problema; en una de un millón, es ruido.

**La columna `únicos` delata cosas.** Si una columna que debería tener cinco
categorías tiene diecisiete valores distintos, hay un problema de escritura. Si una
que debería identificar cada fila tiene menos valores que filas, hay duplicados.

## A2 · Los cinco problemas, contados

Ahora cuantificamos cada uno. Ningún número de acá sale de una estimación: todos se
cuentan.

In [ ]:
print("1. Duplicados exactos      :", tx.duplicated().sum())
print("2. Montos nulos            :", tx["monto"].isna().sum())
print("   Montos negativos o cero :", (tx["monto"] <= 0).sum())
print("3. Grafías de categoría    :", tx["categoria"].nunique(), "valores para 5 rubros reales")
print("4. Huérfanos (id sin CRM)  :", (~tx["id_cliente"].isin(cli["id_cliente"])).sum())
print()
print("Los 17 valores de categoría:")
print(sorted(tx["categoria"].unique()))

1. Duplicados exactos      : 179
2. Montos nulos            : 659
   Montos negativos o cero : 59
3. Grafías de categoría    : 17 valores para 5 rubros reales
4. Huérfanos (id sin CRM)  : 25

Los 17 valores de categoría:
[' Electrónica', ' Hogar', 'Accesorios', 'Accesorios ', 'ELECTRONICA', 'Electrodomesticos', 'Electronica', 'Electronica ', 'HOGAR', 'Hogar', 'INDUMENTARIA', 'Indumentaria', 'Indumentaria ', 'accesorios', 'electronica', 'hogar', 'indumentaria']


Mirá la lista de categorías: `'Electronica'`, `'ELECTRONICA'`, `' Electrónica'`,
`'Electronica '`. Para vos son lo mismo; para pandas son cuatro categorías
distintas, y cualquier `groupby` las va a contar por separado.

Son mayúsculas, espacios al principio o al final, y acentos. Los tres se arreglan
normalizando el texto.

## A3 · Antes de decidir: ¿el nulo tiene patrón?

Esta es la línea de código que cambia la decisión, y casi nadie la corre.

Si los nulos están repartidos parejo, eliminar las filas casi no sesga nada. Si se
concentran en un grupo, eliminarlas **borra ese grupo del análisis**.

In [ ]:
# ¿Los montos nulos se reparten parejo entre canales?
patron = tx.groupby("canal").agg(
    filas      = ("monto", "size"),
    nulos      = ("monto", lambda s: s.isna().sum()),
)
patron["% nulos"] = (patron["nulos"] / patron["filas"] * 100).round(2)
patron

,filas,nulos,% nulos
canal,,,
App,9040,220,2.43
Instagram,2013,47,2.33
Web,12101,321,2.65
WhatsApp,3156,71,2.25


Acá los porcentajes son parecidos entre canales, así que el faltante se comporta
como aleatorio y eliminar es defendible.

**Si en tu dataset un grupo tuviera el triple de nulos que el resto**, ese patrón es
un hallazgo: significa que algún sistema no está registrando bien, y eso va al
informe de calidad y a la brecha.

## A4 · Los valores centinela

Un **centinela** es un valor que un sistema usa para decir «no sé» sin dejar el campo
vacío: `0`, `-1`, `999`, `9999`, `999999`, `1900-01-01`, `"N/A"` escrito a mano.

Son peligrosos porque **no son nulos para la computadora**. `isna()` no los detecta y
entran en los promedios como si fueran datos reales.

In [ ]:
# La clase pasada apareció el id_cliente 999999. ¿Existe en el CRM?
print("¿999999 está en la tabla de clientes?:", (cli["id_cliente"] == 999999).any())
print("Transacciones a nombre de 999999    :", (tx["id_cliente"] == 999999).sum())
print()

# Edades imposibles en el CRM
print("Edades = 0        :", (cli["edad"] == 0).sum())
print("Edades > 100      :", (cli["edad"] > 100).sum())
print("Edades nulas      :", cli["edad"].isna().sum())
print()
print("Si dejáramos las edades imposibles, la media daría:", round(cli["edad"].mean(), 1))
edad_ok = cli["edad"].where((cli["edad"] >= 16) & (cli["edad"] <= 100))
print("Convirtiéndolas en NaN, la media da:      ", round(edad_ok.mean(), 1))

¿999999 está en la tabla de clientes?: False
Transacciones a nombre de 999999    : 25

Edades = 0        : 8
Edades > 100      : 10
Edades nulas      : 80

Si dejáramos las edades imposibles, la media daría: 38.5
Convirtiéndolas en NaN, la media da:       38.4


### Por qué NaN y no cero

Un `0` entra en los promedios como si fuera una edad real. Un `NaN` es honesto: dice
«no sé» y pandas lo excluye solo de los cálculos.

Y borrar al cliente entero tampoco corresponde: sus compras son información válida.
Lo que no sabemos es su edad, no su historial.

> **Ahora mirá el resultado con atención.** La media pasó de 38,5 a 38,4. Casi no
> cambió, porque son 18 valores imposibles sobre 4.000 clientes.
>
> ¿Entonces no valía la pena? **Sí valía, y el punto es justamente ese:** vos no
> sabías que el impacto iba a ser chico hasta que lo mediste. Si hubieran sido 800
> ceros en vez de 18, la media habría caído varios años y todo el análisis por edad
> quedaba torcido.
>
> La regla no es «corregí porque siempre cambia mucho». Es **«medí el impacto de tu
> corrección y decilo»**. Un informe que dice «convertimos 18 edades imposibles en
> nulos; la media apenas se movió de 38,5 a 38,4» es mejor que uno que solo dice
> «limpiamos las edades».

## A5 · La limpieza, con la bitácora armándose sola

Acá está el truco que te va a ahorrar el TP1: **la bitácora se escribe mientras
limpiás**, no después. Cada paso se registra en el momento, con el número de antes.

In [ ]:
tx_limpio = tx.copy()
bitacora = []

def registrar(paso, antes, despues, criterio):
    bitacora.append({
        "paso": paso, "antes": antes, "despues": despues,
        "eliminadas": antes - despues, "criterio": criterio,
    })

# --- 1. Duplicados exactos
n = len(tx_limpio); tx_limpio = tx_limpio.drop_duplicates()
registrar("Duplicados exactos", n, len(tx_limpio),
          "Misma fila completa: doble carga del ETL")

# --- 2. Integridad referencial
n = len(tx_limpio); tx_limpio = tx_limpio[tx_limpio["id_cliente"].isin(cli["id_cliente"])]
registrar("Clientes huérfanos", n, len(tx_limpio),
          "id_cliente sin correspondencia en el CRM")

# --- 3. Normalizar el texto de categoría
tx_limpio["categoria"] = (tx_limpio["categoria"].str.strip().str.lower()
                          .str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("utf-8"))
print("Categorías después de normalizar:", tx_limpio["categoria"].nunique())

# --- 4. Montos nulos o no positivos
n = len(tx_limpio); tx_limpio = tx_limpio[tx_limpio["monto"].notna() & (tx_limpio["monto"] > 0)]
registrar("Montos nulos o ≤ 0", n, len(tx_limpio),
          "No imputamos ventas: una venta imputada no existió")

# --- 5. Outliers por percentil
techo = tx_limpio["monto"].quantile(0.999)
n = len(tx_limpio); tx_limpio = tx_limpio[tx_limpio["monto"] <= techo]
registrar("Outliers de monto", n, len(tx_limpio),
          f"Monto por encima del percentil 99,9 (${techo:,.0f})")

pd.DataFrame(bitacora)

Categorías después de normalizar: 5


,paso,antes,despues,eliminadas,criterio
0,Duplicados exactos,26310,26131,179,Misma fila completa: doble carga del ETL
1,Clientes huérfanos,26131,26106,25,id_cliente sin correspondencia en el CRM
2,Montos nulos o ≤ 0,26106,25395,711,No imputamos ventas: una venta imputada no exi...
3,Outliers de monto,25395,25369,26,"Monto por encima del percentil 99,9 ($2,452,131)"


In [ ]:
retencion = len(tx_limpio) / len(tx)
print(f"Filas originales : {len(tx):,}")
print(f"Filas finales    : {len(tx_limpio):,}")
print(f"Retención        : {retencion:.1%}")

Filas originales : 26,310
Filas finales    : 25,369
Retención        : 96.4%


### Dos cosas de esa tabla

**El orden importa.** Si eliminás outliers antes que duplicados, los números
intermedios no dan lo mismo. Por eso la bitácora lleva el paso en el orden en que se
hizo, no en cualquier orden.

**La última columna es la que vale.** Sin criterio escrito, la tabla es un registro
de lo que pasó; con criterio, es una justificación de lo que decidiste. La rúbrica
del TP1 evalúa la segunda.

> Fijate que retuvimos el 96,4 %. Ese número también va en el informe: dice que la
> limpieza fue quirúrgica y no una masacre. Si hubieras retenido el 60 %, habría que
> explicar muy bien por qué.

---
# PARTE B · Tu dataset

Cinco revisiones, en este orden. Van directo a la sección 4 del TP1.

> **Antes de arrancar:** cargá tu archivo. El código está en la ficha de tu equipo,
> dentro del catálogo.

In [ ]:
import pandas as pd
import numpy as np

# ---- PEGÁ ACÁ LAS LÍNEAS DE CARGA DE TU DATASET ----


# ---- Y COMPLETÁ ESTOS TRES NOMBRES ----
ID    = ""
FECHA = ""
VALOR = ""

print("Cargado:", tx.shape[0], "filas y", tx.shape[1], "columnas")

## Revisión 1 · El informe de calidad

In [ ]:
def informe_calidad(df):
    return pd.DataFrame({
        "tipo":    df.dtypes.astype(str),
        "nulos":   df.isna().sum(),
        "% nulos": (df.isna().mean() * 100).round(2),
        "únicos":  df.nunique(),
    })

informe_calidad(tx)

> **Para el TP1:** esta tabla va completa. Y abajo, dos o tres oraciones que digan
> qué se ve en ella. Una tabla sin lectura no es un informe.

## Revisión 2 · Duplicados

In [ ]:
print("Filas duplicadas exactas:", tx.duplicated().sum())
print()

# ¿Y el identificador de fila? (si tu tabla tiene uno)
# Cambiá el nombre por el de tu columna identificadora.
CLAVE = ID
n, u = len(tx), tx[CLAVE].nunique()
print(f"Filas: {n:,}   |   {CLAVE} distintos: {u:,}")
if u < n:
    print(f"-> {n-u:,} filas comparten identificador.")
    print("   Revisá si son idénticas (calidad) o distintas (granularidad más fina).")

## Revisión 3 · Nulos, y si tienen patrón

In [ ]:
nulos = tx.isna().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)

if len(nulos) == 0:
    print("Ninguna columna tiene nulos declarados.")
else:
    for col, n in nulos.items():
        print(f"  {col:32s} {n:>10,}  ({n/len(tx):.1%})")

In [ ]:
# ¿Los nulos se concentran en algún grupo?
# Elegí una columna categórica de tu dataset (canal, categoría, país, tipo...)
# y una columna que tenga nulos.

CATEGORICA = ""     # ej: "canal", "Region", "category"
CON_NULOS  = ""     # ej: "monto", "Sales", "rating"

if CATEGORICA and CON_NULOS:
    patron = tx.groupby(CATEGORICA).agg(
        filas = (CON_NULOS, "size"),
        nulos = (CON_NULOS, lambda s: s.isna().sum()),
    )
    patron["% nulos"] = (patron["nulos"] / patron["filas"] * 100).round(2)
    display(patron.sort_values("% nulos", ascending=False))
else:
    print("Completá CATEGORICA y CON_NULOS para correr este chequeo.")

> **Para el TP1:** si los porcentajes son parecidos entre grupos, el faltante se
> comporta como aleatorio y eliminar es defendible. Si un grupo tiene el doble o el
> triple, **eso es un hallazgo** y hay que decirlo: significa que algún sistema no
> está registrando bien.

## Revisión 4 · Centinelas y valores imposibles

In [ ]:
# Los mínimos y máximos son donde aparecen los errores de carga
tx.describe().round(2)

In [ ]:
# Los valores más frecuentes de cada columna numérica.
# Si un número redondo aparece muchísimas veces, sospechá: puede ser un centinela.

for col in tx.select_dtypes(include="number").columns[:8]:
    top = tx[col].value_counts().head(3)
    frec = ", ".join(f"{v} ({c:,}x)" for v, c in top.items())
    print(f"{col:28s} -> {frec}")

> **Qué buscar:** ceros donde no deberían existir, `-1`, `999`, `9999`, `999999`,
> fechas de 1900 o 1970, códigos postales en cero, textos como `"N/A"` o `"desconocido"`.
>
> Si encontrás uno, **convertilo en NaN antes de calcular nada** y anotá cuántos eran.
> Esa conversión es la primera línea de tu bitácora.

## Revisión 5 · Tu bitácora

In [ ]:
tx_limpio = tx.copy()
bitacora  = []

def registrar(paso, antes, despues, criterio):
    bitacora.append({
        "paso": paso, "antes": antes, "despues": despues,
        "eliminadas": antes - despues, "criterio": criterio,
    })

# ============================================================
# Escribí acá tus pasos de limpieza. Ejemplo de la estructura:
#
# n = len(tx_limpio); tx_limpio = tx_limpio.drop_duplicates()
# registrar("Duplicados exactos", n, len(tx_limpio),
#           "Misma fila completa: doble carga")
#
# Un bloque de dos líneas por cada decisión. El criterio en castellano,
# explicando POR QUÉ, no qué.
# ============================================================


if bitacora:
    display(pd.DataFrame(bitacora))
    print(f"\nRetención: {len(tx_limpio)/len(tx):.1%}")
else:
    print("Todavía no registraste ningún paso.")

---

## Lo que se llevan de hoy

| Producto | Dónde va en el TP1 |
| --- | --- |
| El informe de calidad, con su lectura | Sección 4 |
| Duplicados y granularidad, verificados | Secciones 3 y 4 |
| Los nulos, y si tienen patrón | Sección 4 |
| Centinelas encontrados | Sección 4 |
| La bitácora completa | Sección 4 |

## Para la clase 5 — es la entrega

El **viernes 11/09 antes del inicio de la clase**, el TP1 completo en PDF a
Classroom. Y la defensa: 6 minutos de exposición, 2 de preguntas. El orden se
sortea ese día, así que vengan todos preparados.

## Los cuatro errores que hunden una bitácora

1. **Escribirla al final.** Reconstruir de memoria no funciona: se pierden los
   números de antes y los criterios se inventan.
2. **Poner el qué sin el por qué.** «Eliminamos 659 filas» no es una bitácora.
3. **No decir cuántas filas.** Sin el antes y el después no se sabe si la decisión
   fue menor o si borró un tercio del dataset.
4. **Limpiar sin medir primero.** Si no contaste antes de tocar, ese número ya no
   existe.

## Si se traban

Por mail, con las cuatro cosas de siempre: qué intentaron, qué esperaban, qué pasó, y
el error completo copiado.